# Running on Quantinuum using predefined workers

In this example we're going beyoind writing our workers.
For this we're going to construct a graph thad runs a `pytket` circuit on a simulator using the `quantinuum_worker`
The worker can be installed using `uv add tkr-quantinuum-worker-impl`

In this example we're also covering a finer grained control over the workflow run.
These are the internals that are set inside the `run_workflow` function.


In [ ]:
from tierkreis.controller.data.models import OpaqueType

Circuit = OpaqueType["pytket._tket.circuit.Circuit"]
BackendResult = OpaqueType["pytket.backends.backendresult.BackendResult"]

Now we can construct a graph using the `quantinuum_worker`.
The api definitions for workers can be imported from the `quantinuum_worker` package which is a dependency we add when installing `tkr-quantinuum-worker-impl`.
We define a graph `Circuit -> BackendResult` using hardcoded information for which emulator backend to use.

In [ ]:
from tierkreis.builder import Graph
from tierkreis.controller.data.models import TKR
from quantinuum_worker import (
    compile_using_info,
    get_backend_info,
    run_circuit,
)

g = Graph(TKR[Circuit], TKR[BackendResult])
info = g.task(get_backend_info(device_name=g.const("H2-1")))
compiled_circuit = g.task(compile_using_info(g.inputs, info))
results = g.task(
    run_circuit(
        circuit=compiled_circuit,
        n_shots=g.const(10),
        device_name=g.const("H2-1SC"),
    ),
)
workflow = g.finish_with_outputs(results)

Now we will define our own storage and executor.
Storage is responsible for setting up the checkpointing; it stores the state of the computation.
We have to provide a uuid, and optionally a name, as before.

In [ ]:
from uuid import UUID

from tierkreis.storage import FileStorage

storage = FileStorage(UUID(int=209), do_cleanup=True, name="quantinuum_submission")

An executor lives in context, where it can access workers to run their `main` entrypoints.
We define this by providing a path to the directory our workers live in, in this case in the `tierkreis_workers` directory.

If you cloned the directory you now have the source of the `quantinuum_worker`.
You could run it using `uv run main.py`.
For this use case we have the `UvExecutor`.

In [ ]:
from tierkreis.consts import PACKAGE_PATH
from tierkreis.executor import UvExecutor

executor = UvExecutor(PACKAGE_PATH.parent / "tierkreis_workers", storage.logs_path)

As an alternative (or if you only installed the worker), the worker exports a script `tkr-quantinuum-worker`.
It functions as a shell binary, hence we can use the `ShellExecutor` for it.
Since using `uv add` will make this script available from within our python environment, we don't need to point to a directory.

In [ ]:
from tierkreis.executor import ShellExecutor

executor = ShellExecutor(registry_path=None, workflow_dir=storage.workflow_dir)

Since this graph is using the `qnexus` api internally you also need to run the following once: 

In [ ]:
from qnexus.client.auth import login

login()

Once we provide the graph inputs we can now run it by providing a storage and an executor.

In [ ]:
from pathlib import Path

from pytket.qasm.qasm import circuit_from_qasm

from tierkreis import run_graph

circuit = circuit_from_qasm(Path().parent / "data" / "ghz_state_n23.qasm")
run_graph(storage, executor, workflow, circuit)

And finally we can print the outputs

In [ ]:
from tierkreis.storage import read_outputs

outputs = read_outputs(g, storage)